## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

### Nhập các thư viện cần thiết:
- os: Thư viện để làm việc với hệ điều hành, như quản lý tệp, thư mục.

- glob: Hỗ trợ tìm kiếm các tệp theo mẫu trong thư mục.

- load_dotenv từ thư viện dotenv: Dùng để tải các biến môi trường từ file .env, thường chứa các thông tin bảo mật như API keys.

- gradio: Một thư viện giúp tạo giao diện web đơn giản để tương tác với mô hình AI.

In [ ]:
# imports for langchain

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter

### Nhập các module từ langchain:
- DirectoryLoader: Dùng để tải tất cả các tài liệu trong một thư mục.

- TextLoader: Dùng để tải nội dung từ các tệp văn bản cụ thể.

- CharacterTextSplitter: Dùng để chia nhỏ văn bản theo ký tự.

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

### Định nghĩa mô hình và cơ sở dữ liệu vector:
- MODEL: Xác định mô hình AI sẽ sử dụng (ở đây là gpt-4o-mini, một phiên bản rút gọn của GPT-4).

- db_name: Tên của cơ sở dữ liệu vector, có thể được dùng để lưu trữ và truy vấn thông tin bằng phương pháp tìm kiếm ngữ nghĩa.

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

### Tải biến môi trường từ file .env
- load_dotenv(override=True): Tải các biến môi trường từ file .env. Nếu một biến đã tồn tại, nó sẽ được ghi đè.

- os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env'):

   - Đặt giá trị của biến môi trường OPENAI_API_KEY.

   - Nếu không tìm thấy trong .env, sẽ sử dụng giá trị mặc định 'your-key-if-not-using-env'.

➡️ Tác dụng: Đảm bảo khóa API của OpenAI được thiết lập trước khi sử dụng các dịch vụ AI.

In [ ]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase
# Thank you Mark D. and Zoya H. for fixing a bug here..

folders = glob.glob("knowledge-base/*")

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

### Đọc tài liệu từ thư mục "knowledge-base"
- glob.glob("knowledge-base/*"): Lấy danh sách tất cả các thư mục con trong knowledge-base/.

- text_loader_kwargs = {'encoding': 'utf-8'}: Đảm bảo tệp văn bản được đọc với mã hóa UTF-8.

- documents = []: Danh sách chứa các tài liệu đã tải.

- Vòng lặp duyệt qua thư mục knowledge-base/:

   1. Lấy tên thư mục (doc_type).

   2. Sử dụng DirectoryLoader để tải tất cả tệp .md trong thư mục.

   3. Thêm thuộc tính doc_type vào metadata của mỗi tài liệu.

   4. Lưu tài liệu vào danh sách documents.

➡️ Tác dụng:

Tự động tải tài liệu từ thư mục knowledge-base/, giúp hệ thống có thể xử lý chúng sau này (ví dụ: sử dụng AI để tìm kiếm thông tin).



In [ ]:
len(documents)

### Kiểm tra số lượng tài liệu đã tải
Lệnh này đơn giản chỉ đếm số lượng tài liệu đã được tải vào danh sách documents.

➡️ Tác dụng: Xác nhận rằng các tài liệu đã được tải thành công.

In [ ]:
documents[24]

### Truy xuất một tài liệu cụ thể
- Lệnh này lấy tài liệu thứ 25 trong danh sách documents (do Python đánh số từ 0).

- Nếu danh sách có ít hơn 25 phần tử, câu lệnh này có thể gây lỗi.

➡️ Tác dụng: Xem nội dung hoặc metadata của một tài liệu cụ thể.

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

### Chia nhỏ tài liệu thành các đoạn nhỏ
- CharacterTextSplitter(chunk_size=1000, chunk_overlap=200):

    - chunk_size=1000: Mỗi đoạn văn bản sẽ có tối đa 1000 ký tự.

    - chunk_overlap=200: Các đoạn có phần trùng lặp 200 ký tự để tránh mất ngữ cảnh.

- split_documents(documents): Chia tất cả các tài liệu trong danh sách documents thành nhiều phần nhỏ.

➡️ Tác dụng:

- Giúp hệ thống xử lý văn bản dễ dàng hơn, đặc biệt khi làm việc với mô hình AI.

- Tránh lỗi khi nhập dữ liệu có kích thước lớn vào các mô hình như GPT.

In [ ]:
len(chunks)

### Kiểm tra số lượng đoạn văn bản sau khi chia nhỏ
Đếm tổng số đoạn văn bản sau khi đã được chia từ documents.

➡️ Tác dụng: Kiểm tra xem quá trình chia nhỏ có hoạt động đúng không

In [ ]:
chunks[6]

In [ ]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

In [ ]:
for chunk in chunks:
    if 'CEO' in chunk.page_content:
        print(chunk)
        print("_________")